# 05c — 第五讲基础习题参考答案（Solutions）

本 Notebook 给出 `05c_exercises.ipynb` 中 8 道基础编程练习的完整参考解答与讲解。

---

## 0. 公共环境准备

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.pipeline import Pipeline

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors, Descriptors3D, rdMolDescriptors

RDLogger.DisableLog("rdApp.*")
plt.rcParams["figure.dpi"] = 110
np.random.seed(0)

def find_file(name):
    for path in (f"../data/{name}", f"data/{name}", name):
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"找不到 {name}")

df = pd.read_csv(find_file("solubility.csv"))
df = df[(df["MolWt"] > 20) & (df["MolWt"] < 800)].sample(2000, random_state=7).reset_index(drop=True)

FEATS = [c for c in df.columns if c not in ("Name", "SMILES", "Solubility")]
X = df[FEATS].copy()
y = df["Solubility"].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=0)

print(f"X: {X.shape[0]} 个分子 × {X.shape[1]} 个描述符")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（solutions/ 子目录与 Colab 通用）
import pathlib, sys

for _p in (pathlib.Path.cwd().parent, pathlib.Path.cwd().parent.parent):
    sys.path.insert(0, str(_p))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 第一部分：特征选择与 SHAP

### 习题 1 参考答案：找出“准常量”描述符

In [ ]:
top_share = (X_tr
             .apply(lambda s: s.value_counts(normalize=True).iloc[0])
             .sort_values(ascending=False))

print("最高频取值占比（降序前 5）：")
print(top_share.head(5).round(3).to_string())

**讲解**：`NumSaturatedRings` 约 82% 的分子取值都是 0，`NumAliphaticRings` 约 75%——这两列对树模型来说几乎是“只有一个取值”的常量。

注意这个判据比 `VarianceThreshold` 的原始方差稳健得多：方差带量纲，`BertzCT`（范围 0–20000）的方差天然比 `FractionCSP3`（范围 0–1）大几个数量级，直接排序毫无意义。

### 习题 2 参考答案：揪出高度共线的描述符对

In [ ]:
corr = X_tr.corr().abs()

high_pairs = [
    (a, b, corr.loc[a, b])
    for i, a in enumerate(corr.columns)
    for b in corr.columns[i + 1:]              # 只遍历上三角
    if corr.loc[a, b] > 0.95
]
high_pairs.sort(key=lambda t: -t[2])

print(f"|r| > 0.95 的描述符对共 {len(high_pairs)} 组：")
for a, b, r in high_pairs:
    print(f"  |r| = {r:.3f}   {a:<22s} <-> {b}")

**讲解**：挑出来的全部是“分子尺寸”的不同说法——`HeavyAtomCount`（重原子数）、`NumValenceElectrons`（价电子数）、`MolWt`（分子量）、`MolMR`（摩尔折射率）、`LabuteASA`（近似表面积）。

其中 `HeavyAtomCount` 与 `NumValenceElectrons` 的 $|r|$ 高达 0.99，因为对以 C、N、O 为主的有机分子来说，每个重原子平均贡献的价电子数几乎是常数。保留其中任意一个即可，**多留的每一个都只会稀释重要性度量**。

### 习题 3 参考答案：用 RFE 把 17 个描述符压到 5 个

In [ ]:
rfe = RFE(
    estimator=RandomForestRegressor(n_estimators=80, random_state=0, n_jobs=-1),
    n_features_to_select=5,
    step=2,
).fit(X_tr, y_tr)

cols5 = list(X.columns[rfe.support_])
print("RFE 选出的 5 个描述符：", cols5)

r2_all = (RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)
          .fit(X_tr, y_tr).score(X_te, y_te))
r2_top5 = (RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1)
           .fit(X_tr[cols5], y_tr).score(X_te[cols5], y_te))

print(f"\n全部 17 个描述符 : 测试集 R2 = {r2_all:.4f}")
print(f"RFE 选出的 5 个   : 测试集 R2 = {r2_top5:.4f}")
print(f"差值 = {r2_top5 - r2_all:+.4f}")

**讲解**：5 个特征的 $R^2$ 不但没掉，甚至**略高于**全部 17 个。为什么减少特征反而可能提升泛化？

1. **降低方差**：随机森林每次分裂时从 `max_features` 个随机候选里挑最优。候选池里塞满冗余特征，相当于稀释了真正有用特征被抽到的概率；
2. **减少噪声拟合**：准常量和高噪声特征偶尔会在训练集上制造“伪分裂”，这些分裂在测试集上纯属噪声；
3. **共线性解耦**：17 个特征里有 6 对 $|r| > 0.95$，它们携带的信息在模型里被反复重复计算。

当然，这个差值（约 0.008）本身落在随机波动范围内。**正确的说法是“精度不变而维度减少了三分之二”，而不是“降维提升了精度”**。

### 习题 4 参考答案：验证“选择性数据泄漏”

In [ ]:
cv = KFold(5, shuffle=True, random_state=0)

def rf_eval():
    return RandomForestRegressor(n_estimators=120, random_state=0, n_jobs=-1)

def rf_sel():
    return RandomForestRegressor(n_estimators=80, random_state=0, n_jobs=-1)

# 写法 A（有泄漏）：先在全量 (X, y) 上选特征
leaky_sel  = RFE(rf_sel(), n_features_to_select=4, step=3).fit(X, y)
leaky_cols = X.columns[leaky_sel.support_]
score_A = cross_val_score(rf_eval(), X[leaky_cols], y, cv=cv, scoring="r2").mean()

# 写法 B（正确）：特征选择进 Pipeline，每折独立重选
pipe = Pipeline([
    ("select", RFE(rf_sel(), n_features_to_select=4, step=3)),
    ("model",  rf_eval()),
])
score_B = cross_val_score(pipe, X, y, cv=cv, scoring="r2").mean()

print(f"A（全量选特征后再 CV）: R2 = {score_A:.4f}")
print(f"B（选特征进 Pipeline）: R2 = {score_B:.4f}")
print(f"\n乐观偏差 A - B = {score_A - score_B:+.4f}")
print("入选特征 (A):", list(leaky_cols))

**讲解**：在这个数据集上两者几乎完全相同，甚至可能出现 A 略低于 B 的情况。原因是：

- 2,000 个样本 vs 17 个候选特征，**样本/特征比高达 100 以上**；
- `MolLogP` 与 $\log S$ 的相关性极强，任何一折的子集都会把它选进来。

换句话说，“选哪 4 个特征”这个决定在这里**几乎不含可泄漏的信息**——每折选出来的都是同一批。

**危险场景恰恰相反**：假设你只有 40 个实测器件样本，却算了 1,800 维 mordred 描述符。此时总能找到某些描述符**纯粹由于随机巧合**与这 40 个标签高度相关。在全量数据上选中它们、再去做交叉验证，验证折的标签已经参与过“这些特征很厉害”这个判断，$R^2$ 可以被吹到 0.8 以上——而换一批新样本就原形毕露。**样本越少、候选特征越多，这个偏差就越致命。**

### 习题 5 参考答案：手算两个特征的精确 Shapley 值

In [ ]:
A, B = "MolLogP", "TPSA"

m_A  = RandomForestRegressor(n_estimators=80, random_state=0, n_jobs=-1).fit(X_tr[[A]], y_tr)
m_B  = RandomForestRegressor(n_estimators=80, random_state=0, n_jobs=-1).fit(X_tr[[B]], y_tr)
m_AB = RandomForestRegressor(n_estimators=80, random_state=0, n_jobs=-1).fit(X_tr[[A, B]], y_tr)

x_demo = X_te[[A, B]].iloc[[0]]
print("待解释样本：")
print(x_demo.to_string(index=False))

# 1) 四个子集的价值 v(S)
v_empty = y_tr.mean()                 # v(空集) = 训练集均值
v_A     = m_A.predict(x_demo[[A]])[0]
v_B     = m_B.predict(x_demo[[B]])[0]
v_AB    = m_AB.predict(x_demo)[0]

# 2) 两特征情形下的 Shapley 值
phi_A = 0.5 * (v_A - v_empty) + 0.5 * (v_AB - v_B)
phi_B = 0.5 * (v_B - v_empty) + 0.5 * (v_AB - v_A)

print(f"\nv(empty)       = {v_empty:+.4f}")
print(f"v({A})     = {v_A:+.4f}")
print(f"v({B})        = {v_B:+.4f}")
print(f"v({A},{B}) = {v_AB:+.4f}")
print(f"\nphi({A})   = {phi_A:+.4f}")
print(f"phi({B})      = {phi_B:+.4f}")

# 3) 有效性公理
lhs = v_empty + phi_A + phi_B
print(f"\n有效性公理：v(empty) + phi_A + phi_B = {lhs:.10f}")
print(f"            v(A, B)                  = {v_AB:.10f}")
print(f"            残差                      = {abs(lhs - v_AB):.2e}")

**讲解**：残差是 0 或机器精度量级——有效性公理**精确成立**，这是代数恒等式而非数值近似。把两式相加可以直接看出来：

$$\phi_A + \phi_B = \tfrac{1}{2}\big[v_A - v_\varnothing + v_{AB} - v_B\big] + \tfrac{1}{2}\big[v_B - v_\varnothing + v_{AB} - v_A\big] = v_{AB} - v_\varnothing$$

中间项两两抵消，剩下的正是“完整模型预测 − 基准值”。

从数值上看：`MolLogP` 的贡献为负（把预测推向更难溶），`TPSA` 的贡献为正（极性表面积大，推向更易溶）——与化学直觉完全一致。

**注意这里的 $v(S)$ 用的是“重训模型”口径**，与 `shap.TreeExplainer` 的 TreeSHAP 口径不同（后者不重训模型，而是在树里对缺失特征做加权下行）。两者在特征独立时结果接近，在特征强相关时会有系统性差异——这是 SHAP 文献里长期争论的 interventional 与 conditional 之分。

---

## 第二部分：三维构象与 PubChem API

### 习题 6 参考答案：生成三维构象并读出 MMFF 能量

In [ ]:
smi_anthracene = "c1ccc2cc3ccccc3cc2c1"

mol = Chem.MolFromSmiles(smi_anthracene)
n_before = mol.GetNumAtoms()

# 1) 加氢（关键的一步）
mol = Chem.AddHs(mol)

# 2) ETKDG 嵌入
params = AllChem.ETKDGv3()
params.randomSeed = 42
code = AllChem.EmbedMolecule(mol, params)
print(f"ETKDG 嵌入返回码 = {code}（0 表示成功）")

# 3) MMFF 优化并读取能量
AllChem.MMFFOptimizeMolecule(mol, maxIters=1000)
ff = AllChem.MMFFGetMoleculeForceField(mol, AllChem.MMFFGetMoleculeProperties(mol))
print(f"MMFF 能量 = {ff.CalcEnergy():.2f} kcal/mol")

# 4) 原子数对比
print(f"\n加氢前原子数 = {n_before}（仅重原子，C14）")
print(f"加氢后原子数 = {mol.GetNumAtoms()}（C14H10）")

**讲解**：蒽的分子式是 $\text{C}_{14}\text{H}_{10}$，所以加氢后原子数从 14 变成 24，与化学式吻合。

**为什么 `AddHs` 不能省？** RDKit 的 SMILES 解析默认把氢原子“隐式”存在重原子的属性里，不占坐标。如果不加氢直接嵌入：

- 碳原子周围没有氢，力场算不出正确的空间位阻与 $sp^3$ 四面体构型；
- 立体化学（R/S、顺反）无法正确表达；
- 所有 3D 描述符（回转半径、表面积、PBF）都会系统性偏小。

**这是 RDKit 三维操作里最常见的一个错误**，而且它不会报错，只会悄悄给出错误的数字。

另外，蒽是完全刚性的平面分子，所以无论用什么随机种子，嵌入出来的构象都几乎相同，MMFF 能量也高度稳定。

### 习题 7 参考答案：用 NPR1 / NPR2 判断分子形状

In [ ]:
def lowest_energy_conformer(smi, n_confs=10, seed=42):
    """生成 n_confs 个构象并 MMFF 优化，返回 (mol, 能量最低构象的 confId)。"""
    mol = Chem.AddHs(Chem.MolFromSmiles(smi))
    ps = AllChem.ETKDGv3()
    ps.randomSeed = seed
    AllChem.EmbedMultipleConfs(mol, numConfs=n_confs, params=ps)
    res = AllChem.MMFFOptimizeMoleculeConfs(mol, maxIters=600)
    return mol, int(np.argmin([e for _c, e in res]))

SHAPES = {
    "Hexane (rod)":        "CCCCCC",
    "Pentacene (disc)":    "c1ccc2cc3cc4cc5ccccc5cc4cc3cc2c1",
    "Adamantane (sphere)": "C1C2CC3CC1CC(C2)C3",
}

rows = []
for name, smi in SHAPES.items():
    mol, cid = lowest_energy_conformer(smi)
    rows.append({
        "molecule": name,
        "NPR1": Descriptors3D.NPR1(mol, confId=cid),
        "NPR2": Descriptors3D.NPR2(mol, confId=cid),
        "PBF":  rdMolDescriptors.CalcPBF(mol, confId=cid),
    })

shape_tbl = pd.DataFrame(rows).set_index("molecule")
print(shape_tbl.round(3).to_string())

**回答**

**1. 哪个分子的 `NPR1` 最接近 1.0？** 金刚烷（$\text{NPR1} \approx \text{NPR2} \approx 1.0$）。

$\text{NPR1} = I_1/I_3$，$\text{NPR2} = I_2/I_3$。当三个主惯性矩完全相等（$I_1 = I_2 = I_3$）时两者都等于 1，对应形状三角的 **球状（sphere）顶点** $(1, 1)$。金刚烷是高度对称的笼状分子，三个方向的尺寸几乎一样，所以落在这个顶点上。

正己烷和并五苯的 `NPR1` 都在 0.08 左右——它们都是**细长**的分子（$I_1 \ll I_3$），靠近 rod–disc 那条边。要区分这两者，光看 NPR 不够，还得看 `PBF`。

**2. 哪个分子的 `PBF` 最接近 0？** 并五苯（$\text{PBF} \approx 0.00$）。

`PBF`（Plane of Best Fit）的定义是：**先对全部原子拟合一个最佳平面，再算所有原子到这个平面的平均垂直距离**。

- 并五苯是完全共轭的平面稠环，所有碳原子严格共面，因此 `PBF` 约等于 0；
- 正己烷的 all-anti 构象虽然骨架是锯齿形平面链，但氢原子会伸出平面，`PBF` 约 0.5 Å；
- 金刚烷是三维笼状结构，`PBF` 超过 1 Å。

**这就是为什么 `PBF` 在有机半导体领域格外有用**：$\pi$–$\pi$ 堆积要求分子足够平面，`PBF` 可以作为虚拟筛选时一个直接、廉价的平面性过滤器。

### 习题 8 参考答案：用 PubChem 抓取数据并验证 Yalkowsky 方程

In [ ]:
PUG_REST = "https://pubchem.ncbi.nlm.nih.gov/rest/pug"
ACENE_CIDS = {"naphthalene": 931, "anthracene": 8418, "phenanthrene": 995,
              "pyrene": 31423, "chrysene": 9171}

mp_df = pd.read_csv(find_file("melting_points.csv"))

# 1) 一次请求拿回 5 个分子的性质
try:
    url = (f"{PUG_REST}/compound/cid/"
           + ",".join(str(c) for c in ACENE_CIDS.values())
           + "/property/MolecularWeight,XLogP/JSON")
    r = requests.get(url, timeout=25)
    time.sleep(0.25)                       # 遵守 <= 5 req/s 的限速
    props = pd.DataFrame(r.json()["PropertyTable"]["Properties"])
    props["name"] = props["CID"].map({v: k for k, v in ACENE_CIDS.items()})
    print("PubChem 返回：")
    print(props.to_string(index=False))
except Exception as exc:
    print("网络不可用，改用本地 MolLogP：", exc)
    props = (mp_df[mp_df["name"].isin(ACENE_CIDS)][["name", "MolLogP"]]
             .rename(columns={"MolLogP": "XLogP"}))

# 2) 与本地熔点 / 实测 logS 合并
tbl = props.merge(mp_df[["name", "Tm_C", "logS"]], on="name")

# 3) 代入 Yalkowsky 方程
tbl["logS_pred"] = 0.5 - tbl["XLogP"] - 0.01 * (tbl["Tm_C"] - 25)
tbl["error"] = tbl["logS_pred"] - tbl["logS"]
rmse = np.sqrt((tbl["error"] ** 2).mean())

print("\nYalkowsky GSE 预测 vs 实测：")
print(tbl[["name", "XLogP", "Tm_C", "logS_pred", "logS", "error"]]
      .round(3).to_string(index=False))
print(f"\nRMSE = {rmse:.3f} 个对数单位")

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5))
ax.scatter(tbl["logS"], tbl["logS_pred"], s=110, color=C.crimson, zorder=3)
for _, r in tbl.iterrows():
    ax.annotate(r["name"], (r["logS"], r["logS_pred"]),
                textcoords="offset points", xytext=(8, -4), fontsize=9)
lims = [-9, -2]
ax.plot(lims, lims, "--", color="#888888", lw=1.3, label="y = x")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("Measured logS")
ax.set_ylabel("Yalkowsky GSE predicted logS")
ax.set_title(f"GSE on the Acene Series (RMSE = {rmse:.3f})")
ax.legend()
plt.show()

**讲解**：5 个并苯分子的 RMSE 低于 1 个对数单位。考虑到这个公式**只用了两个变量、三个固定常数、没有做任何拟合**，这个精度相当惊人——作为对照，`05a` 里训练了 200 棵树、用了 10 个描述符的随机森林，在整个 AqSolDB 上的 RMSE 也在 1 个对数单位附近。

注意观察 `error` 这一列：**5 个误差全部为正**，即 GSE 系统性地**略微高估**了并苯的溶解度。这不是偶然，背后有明确的物理原因：

1. **并苯的晶格能被低估**：GSE 用 $-0.01\,(T_m - 25)$ 这个线性项近似晶格破坏能。并苯类的 herringbone 堆积有额外的 $\pi$–$\pi$ 色散稳定化，实际晶格能比线性外推的更大；
2. **PubChem 的 `XLogP` 是计算值**：`XLogP3` 本身就是一个预测模型的输出，对高度疏水的稠环体系往往略微低估——而 $\log P$ 低估会直接导致 $\log S$ 高估。

有意思的是，**正是这种系统性偏差给机器学习留下了用武之地**：如果 GSE 的残差是纯随机噪声，那任何模型都无法再改进；但既然残差与分子类别（稠环 vs 柔性链）系统相关，模型就有可能学到这个修正项。这也是现代溶解度预测模型（如第十讲的 Chemprop D-MPNN）相对 GSE 的主要增益来源。